# Gloms: OME-Zarr -> precomputed raster + meshes -> Neuroglancer
This is the simplest example -- it downloads a real, published OME-Zarr v0.4 dataset (glomeruli segmentation) and converts it directly to precomputed raster + meshes in one combined function call, since OME-Zarr is already chunked and doesn't need to go through a SpatialData staging step first (unlike OME-TIFF -- see the melanoma/invasive notebooks for that path).


In [ ]:
from pathlib import Path
import subprocess
import hashlib
from dirhash import dirhash

from tissue_map_tools.igneous_converters import (
    from_ome_zarr_04_raster_to_sharded_precomputed_raster_and_meshes,
)
from tissue_map_tools.view import view_precomputed_in_neuroglancer, view_precomputed_in_vitessce

out_path = Path.cwd().parent.parent / 'out'
out_path.mkdir(exist_ok=True)
precomputed_path = out_path / 'gloms_precomputed'

## 1. Download and unzip the published dataset (checksum-guarded, safe to re-run)

In [ ]:
URL = 'https://s3.embl.de/spatialdata/raw_data/20_1_gloms.zip'
CHECKSUM_DOWNLOAD = '7857a41d9d4d2914353c9ad0f4ea4ede'
CHECKSUM_UNZIPPED = '927146f7a8cbfcbf9de047a6e1e71226'

download_path = out_path / Path(URL).name
unzipped_path = out_path / Path(URL).stem

if (
    not download_path.exists()
    or CHECKSUM_DOWNLOAD != hashlib.md5(download_path.read_bytes()).hexdigest()
):
    subprocess.run(f'curl -o "{download_path}" "{URL}"', shell=True, check=True)

if not unzipped_path.exists() or CHECKSUM_UNZIPPED != dirhash(unzipped_path, 'md5'):
    subprocess.run(f'unzip -o "{download_path}" -d "{out_path}"', shell=True, check=True)

## 2. Convert directly (OME-Zarr -> precomputed raster + meshes, one call)

In [ ]:
if not (precomputed_path / 'info').exists():
    from_ome_zarr_04_raster_to_sharded_precomputed_raster_and_meshes(
        ome_zarr_path=str(unzipped_path / '0'),
        precomputed_path=str(precomputed_path),
    )
    print('Conversion complete.')
else:
    print('Precomputed output already exists -- skipping conversion.')

## 3. View

This will open with Neuroglancer's own default framing for the data, you can interact with the view directly.

In [ ]:
# viewer = view_precomputed_in_neuroglancer(data_path=str(precomputed_path))
# viewer

In [ ]:
initial_camera_state={
    "position": [0, 0, 0],
    "projectionScale": 5466806.071355488,
    "projectionOrientation": [
        -0.636204183101654,
        -0.5028395652770996,
        0.5443811416625977,
        0.2145828753709793,
    ],
}
viewer = view_precomputed_in_vitessce(data_path=str(precomputed_path), initial_camera_state=initial_camera_state, use_web_app=True)
viewer